# Notebook 14 — Runner Counts, Numbers and Entries

## Purpose

This notebook investigates the source fields `ran` and `num` before they are used in the future database.

It does not assume that:

- `ran` always means the number of starters;
- `ran` is constant within every race;
- source runner rows always equal `ran`;
- `num` is unique within a race;
- `num` identifies a horse or runner consistently across jurisdictions;
- zero or blank `num` values have one universal meaning; or
- either field is suitable for use as a database key.

The investigation will establish:

1. how `ran` and `num` are stored;
2. their missing, zero and value patterns;
3. whether `ran` is internally consistent at race level;
4. how source row counts compare with `ran`;
5. the meaning of the five known races where source rows are below `ran`;
6. duplicate, coupled and suffixed runner-number conventions;
7. jurisdiction, year and race-type differences;
8. safe staging representations, statuses and validation rules.

Provisional races are identified using:

`date + course + off`

The supplied `race_id` remains source lineage and is not treated as a unique race key.

The leading candidate runner-record identity remains:

`candidate race identity + horse`

`num` remains a source attribute unless this notebook establishes a narrower governed use.

All raw-source queries must:

- use read-only SQLite access;
- query the `data` table; and
- exclude the header-like first row with `rowid <> 1`.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

assert DB_PATH.exists(), f"Source database not found: {DB_PATH}"

connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

schema = pd.read_sql_query("PRAGMA table_info(data)", connection)
schema

,cid,name,type,notnull,dflt_value,pk
0,0,date,NUMERIC,0,None,0
1,1,course,TEXT,0,None,0
2,2,race_id,INTEGER,0,None,0
3,3,off,TEXT,0,None,0
4,4,race_name,TEXT,0,None,0
5,5,type,TEXT,0,None,0
6,6,class,TEXT,0,None,0
7,7,pattern,TEXT,0,None,0
8,8,rating_band,TEXT,0,None,0
9,9,age_band,TEXT,0,None,0


## Stage 1 — Raw storage and basic coverage

The table schema declares both `ran` and `num` as `INTEGER`.

SQLite uses dynamic typing, so this does not guarantee that every stored value is an integer. This stage profiles:

- actual SQLite storage classes;
- null values;
- text blanks;
- zero values;
- distinct values; and
- observed numeric ranges.

No semantic interpretation is made yet.

In [2]:
# Profile the two source fields at runner-row level.
#
# The table declares both fields as INTEGER, but SQLite can still store values
# using other storage classes. We therefore inspect the actual stored types
# rather than relying on the declared schema.

field_profile = pd.read_sql_query(
    """
    WITH source AS (
        SELECT
            ran,
            num,
            typeof(ran) AS ran_storage_class,
            typeof(num) AS num_storage_class
        FROM data
        WHERE rowid <> 1
    )
    SELECT
        COUNT(*) AS runner_rows,

        -- `ran` coverage and range.
        SUM(ran IS NULL) AS ran_null_rows,
        SUM(
            typeof(ran) = 'text'
            AND trim(CAST(ran AS TEXT)) = ''
        ) AS ran_blank_text_rows,
        SUM(ran = 0) AS ran_zero_rows,
        COUNT(DISTINCT ran) AS ran_distinct_values,
        MIN(
            CASE
                WHEN typeof(ran) IN ('integer', 'real') THEN ran
            END
        ) AS ran_numeric_min,
        MAX(
            CASE
                WHEN typeof(ran) IN ('integer', 'real') THEN ran
            END
        ) AS ran_numeric_max,

        -- `num` coverage and range.
        SUM(num IS NULL) AS num_null_rows,
        SUM(
            typeof(num) = 'text'
            AND trim(CAST(num AS TEXT)) = ''
        ) AS num_blank_text_rows,
        SUM(num = 0) AS num_zero_rows,
        COUNT(DISTINCT num) AS num_distinct_values,
        MIN(
            CASE
                WHEN typeof(num) IN ('integer', 'real') THEN num
            END
        ) AS num_numeric_min,
        MAX(
            CASE
                WHEN typeof(num) IN ('integer', 'real') THEN num
            END
        ) AS num_numeric_max
    FROM source
    """,
    connection,
)

# Count the actual SQLite storage classes used by each field.
#
# This is separate from the summary above so that unexpected text, real, blob
# or null storage is visible rather than hidden inside aggregate totals.

storage_classes = pd.read_sql_query(
    """
    SELECT
        'ran' AS field,
        typeof(ran) AS storage_class,
        COUNT(*) AS runner_rows
    FROM data
    WHERE rowid <> 1
    GROUP BY typeof(ran)

    UNION ALL

    SELECT
        'num' AS field,
        typeof(num) AS storage_class,
        COUNT(*) AS runner_rows
    FROM data
    WHERE rowid <> 1
    GROUP BY typeof(num)

    ORDER BY field, runner_rows DESC
    """,
    connection,
)

display(field_profile)
display(storage_classes)

,runner_rows,ran_null_rows,ran_blank_text_rows,ran_zero_rows,ran_distinct_values,ran_numeric_min,ran_numeric_max,num_null_rows,num_blank_text_rows,num_zero_rows,num_distinct_values,num_numeric_min,num_numeric_max
0,1851285,0,0,0,37,1,40,0,7032,1179,42,0,40


,field,storage_class,runner_rows
0,num,integer,1844253
1,num,text,7032
2,ran,integer,1851285


### Initial finding

The two fields have very different raw-storage behaviour.

`ran` is structurally clean:

- every source row stores an integer;
- there are no null, blank or zero values;
- values range from 1 to 40.

This does not yet prove that `ran` is semantically correct or constant within
each race, but it means no parsing or raw-value normalisation is required.

`num` uses three observable raw states:

- positive integer values;
- integer zeroes; and
- text blanks.

All 7,032 text values are blank strings. There is therefore no evidence at this
stage that coupled-entry suffixes such as `1A` are stored directly in `num`.
Such conventions may instead have been flattened, omitted or represented in
another field.

The next step is to inspect the frequency distribution of both fields and
confirm that no unexpected values are hidden within the summary counts.

In [3]:
# Inspect the complete value distribution for `ran`.
#
# There are only 37 distinct values, so showing the full distribution is more
# useful than sampling. This will reveal whether unusual field sizes are rare
# isolated cases or established parts of the source.

ran_value_distribution = pd.read_sql_query(
    """
    SELECT
        ran,
        typeof(ran) AS storage_class,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT date || '|' || course || '|' || off) AS provisional_races
    FROM data
    WHERE rowid <> 1
    GROUP BY
        ran,
        typeof(ran)
    ORDER BY
        ran
    """,
    connection,
)

# Inspect every raw state used by `num`.
#
# Text blanks are labelled explicitly so that they remain distinguishable from
# integer zeroes. Casting without checking the storage class could otherwise
# collapse the two source states.

num_value_distribution = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN typeof(num) = 'text'
                 AND trim(CAST(num AS TEXT)) = ''
                THEN '[blank text]'
            ELSE CAST(num AS TEXT)
        END AS raw_num,
        typeof(num) AS storage_class,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT date || '|' || course || '|' || off) AS provisional_races
    FROM data
    WHERE rowid <> 1
    GROUP BY
        num,
        typeof(num)
    ORDER BY
        CASE
            WHEN typeof(num) = 'text' THEN 0
            ELSE 1
        END,
        CASE
            WHEN typeof(num) IN ('integer', 'real') THEN num
        END
    """,
    connection,
)

display(ran_value_distribution)
display(num_value_distribution)

,ran,storage_class,runner_rows,provisional_races
0,1,integer,22,22
1,2,integer,688,344
2,3,integer,5949,1983
3,4,integer,23856,5964
4,5,integer,58098,11620
5,6,integer,98636,16440
6,7,integer,138894,19842
7,8,integer,167367,20921
8,9,integer,179289,19921
9,10,integer,181640,18164


,raw_num,storage_class,runner_rows,provisional_races
0,[blank text],text,7032,877
1,0,integer,1179,186
2,1,integer,175522,175386
3,2,integer,176109,176029
4,3,integer,175925,175865
5,4,integer,175151,175105
6,5,integer,171354,171302
7,6,integer,163013,162981
8,7,integer,150581,150546
9,8,integer,134371,134349


### Value-distribution finding

The complete distributions confirm that no additional raw formats are hidden
inside either field.

`ran` contains 37 integer values between 1 and 40. Values 35, 36 and 37 do not
occur. In most of the distribution, the number of runner rows is exactly the
reported field size multiplied by the number of provisional races carrying
that value. This is consistent with `ran` being repeated on every runner row
within a race.

That relationship must now be tested directly rather than inferred from the
aggregate distribution.

`num` contains:

- 7,032 blank-text rows across 877 provisional races;
- 1,179 integer-zero rows across 186 provisional races; and
- positive integers from 1 to 40.

There are no textual suffixes such as `1A` stored directly in `num`.

For several positive values, the runner-row count exceeds the number of races
in which the value appears. This establishes that at least some provisional
races contain the same positive `num` more than once. It does not yet establish
why those duplicates occur.

The next stage tests `ran` at provisional-race level and compares it directly
with the number of source runner rows.

In [4]:
# Collapse the runner-level source into one record per provisional race.
#
# We retain the minimum, maximum and distinct count of `ran` so that a race
# with inconsistent runner-row values cannot be hidden by selecting only one
# row. The source row count is then compared with the reported value.

race_ran_profile = pd.read_sql_query(
    """
    SELECT
        date,
        course,
        off,
        MIN(ran) AS min_ran,
        MAX(ran) AS max_ran,
        COUNT(DISTINCT ran) AS distinct_ran_values,
        COUNT(*) AS source_runner_rows
    FROM data
    WHERE rowid <> 1
    GROUP BY
        date,
        course,
        off
    """,
    connection,
)

# Classify each provisional race without assuming in advance that `ran` is
# internally consistent.
#
# A direct comparison is only valid where every row in the provisional race
# carries the same `ran` value.

race_ran_profile["ran_consistency_status"] = race_ran_profile.apply(
    lambda row: (
        "consistent"
        if row["distinct_ran_values"] == 1
        else "inconsistent"
    ),
    axis=1,
)

race_ran_profile["row_count_difference"] = (
    race_ran_profile["source_runner_rows"] - race_ran_profile["max_ran"]
)

race_ran_profile["row_count_status"] = race_ran_profile.apply(
    lambda row: (
        "not_comparable_inconsistent_ran"
        if row["distinct_ran_values"] != 1
        else "source_rows_equal_ran"
        if row["row_count_difference"] == 0
        else "source_rows_below_ran"
        if row["row_count_difference"] < 0
        else "source_rows_above_ran"
    ),
    axis=1,
)

# Summarise the source-wide race-level result.

ran_consistency_summary = (
    race_ran_profile
    .groupby(
        [
            "ran_consistency_status",
            "row_count_status",
        ],
        dropna=False,
    )
    .agg(
        provisional_races=("date", "size"),
        source_runner_rows=("source_runner_rows", "sum"),
        minimum_difference=("row_count_difference", "min"),
        maximum_difference=("row_count_difference", "max"),
    )
    .reset_index()
    .sort_values(
        [
            "ran_consistency_status",
            "row_count_status",
        ]
    )
)

display(ran_consistency_summary)

,ran_consistency_status,row_count_status,provisional_races,source_runner_rows,minimum_difference,maximum_difference
0,consistent,source_rows_below_ran,5,32,-4,-1
1,consistent,source_rows_equal_ran,189038,1851253,0,0


## Stage 2 — Race-level consistency of `ran`

The race-level test produces a very strong result.

Across all 189,043 provisional races:

- every runner row within a race carries the same `ran` value;
- 189,038 races have exactly as many source rows as the reported `ran`;
- five races have fewer source rows than the reported `ran`;
- no race has more source rows than `ran`.

The five exceptions contain 32 surviving source rows and are short by between
one and four rows.

This establishes that `ran` is not merely a loosely populated runner attribute.
It behaves as a repeated race-level count throughout the source.

However, the result does not yet establish whether `ran` means:

- declared runners;
- final starters;
- classified finishers;
- all participants including non-finishers; or
- the expected number of source records before rows were lost.

The five exceptions must therefore be inspected individually before assigning
a governed semantic meaning.

In [5]:
# Isolate the five provisional races where the number of surviving source rows
# is lower than the repeated `ran` value.
#
# We first produce a compact race-level summary. Race name, race type and the
# supplied race_id are retained as descriptive and lineage fields, but they are
# not used as part of the provisional race key.

below_ran_races = pd.read_sql_query(
    """
    WITH race_profile AS (
        SELECT
            date,
            course,
            off,
            MIN(race_id) AS minimum_race_id,
            MAX(race_id) AS maximum_race_id,
            MIN(race_name) AS race_name,
            MIN(type) AS race_type,
            MIN(ran) AS reported_ran,
            MAX(ran) AS maximum_ran,
            COUNT(DISTINCT ran) AS distinct_ran_values,
            COUNT(*) AS source_runner_rows
        FROM data
        WHERE rowid <> 1
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        date,
        course,
        off,
        minimum_race_id,
        maximum_race_id,
        race_name,
        race_type,
        reported_ran,
        source_runner_rows,
        source_runner_rows - reported_ran AS row_count_difference
    FROM race_profile
    WHERE
        distinct_ran_values = 1
        AND source_runner_rows < reported_ran
    ORDER BY
        date,
        course,
        off
    """,
    connection,
)

display(below_ran_races)

,date,course,off,minimum_race_id,maximum_race_id,race_name,race_type,reported_ran,source_runner_rows,row_count_difference
0,2024-06-18,Nantes (FR),2:14,873374,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,-1
1,2024-06-26,Ohi (JPN),11:07,871722,871722,Teio Sho (Local (Dirt),Flat,5,4,-1
2,2024-09-03,Morioka (JPN),11:07,876244,876244,Kozukata Sho (Local (Dirt),Flat,5,4,-1
3,2024-09-26,Funabashi (JPN),11:07,878244,878244,Marine Cup (Local (Fillies) (Dirt),Flat,6,2,-4
4,2025-10-09,Ohi (JPN),11:07,905803,905803,Tokyo Hai (Local (Dirt),Flat,16,15,-1


### The five races below `ran`

The five exceptions are not evenly distributed across the source:

- four are Japanese local races;
- three of those four share the raw off-time `11:07`;
- all four Japanese exceptions occur from June 2024 onward;
- the remaining exception is a French hurdle at Nantes.

This clustering does not prove a common cause. It does make a generic interpretation
such as “`ran` counts declarations while the source stores starters” less convincing,
because that distinction would be expected to occur much more widely.

The next step inspects every surviving runner row from these races. The aim is to
determine whether the missing records can be inferred from:

- gaps in `num`;
- gaps in numeric finishing positions;
- non-finisher outcomes;
- comments or beaten-distance patterns;
- duplicated numbers; or
- evidence that only selected runners were imported.

In [6]:
# Retrieve every surviving runner row from the five races where source rows
# fall below the repeated `ran` value.
#
# The output includes fields that may expose a missing runner indirectly:
#
# - `num` may reveal a gap in the card-number sequence;
# - `pos` may reveal a missing finishing position;
# - `ovr_btn` and `btn` may help distinguish finishers from non-finishers;
# - `horse`, connections and comments may reveal whether the source retained
#   only selected runners or omitted a particular outcome category.
#
# `source_rowid` is preserved for physical source lineage and reproducibility.

below_ran_runner_rows = pd.read_sql_query(
    """
    WITH affected_races AS (
        SELECT
            date,
            course,
            off
        FROM data
        WHERE rowid <> 1
        GROUP BY
            date,
            course,
            off
        HAVING
            COUNT(DISTINCT ran) = 1
            AND COUNT(*) < MAX(ran)
    )
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.off,
        d.race_id,
        d.race_name,
        d.type AS race_type,
        d.ran,
        d.num,
        typeof(d.num) AS num_storage_class,
        d.pos,
        typeof(d.pos) AS pos_storage_class,
        d.ovr_btn,
        d.btn,
        d.horse,
        d.jockey,
        d.trainer,
        d.sp,
        d.comment
    FROM data AS d
    INNER JOIN affected_races AS a
        ON d.date = a.date
        AND d.course = a.course
        AND d.off = a.off
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        CASE
            WHEN typeof(d.pos) = 'integer' THEN d.pos
            ELSE 999
        END,
        CASE
            WHEN typeof(d.num) = 'integer' THEN d.num
            ELSE 999
        END,
        d.rowid
    """,
    connection,
)

display(below_ran_runner_rows)

,source_rowid,date,course,off,race_id,race_name,race_type,ran,num,num_storage_class,pos,pos_storage_class,ovr_btn,btn,horse,jockey,trainer,sp,comment
0,1524983,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,,text,1,integer,0.00,0.00,Pyrrhaa (FR),Paulin Blot,Mme Brigitte Re-Scandella,74/10,
1,1524982,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,,text,2,integer,2.50,2.50,Lhubert De Houelle (FR),Mme Cathy Joubert,A Chaille-Chaille,58/10,
2,1524981,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,,text,3,integer,5.00,2.50,Bakarelo (IRE),Leo-Paul Brechet,Gabriel Leenders,7/5F,
3,1524980,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,,text,4,integer,11.00,6.00,Shalez (FR),Gaetan Champier,Mme Brigitte Re-Scandella,19/1,
4,1524979,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,,text,5,integer,13.00,2.00,Bonham Strand (FR),Francesco Mula,Francois Nicolle,4/1,
5,1524978,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,,text,6,integer,23.00,10.00,Lulu Stars (FR),Gwen Richard,Mme V Seignoux,22/1,
6,1524977,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,,text,7,integer,53.00,30.00,Cockling (FR),Stephane Paillard,J Jouin,25/1,
7,1528630,2024-06-26,Ohi (JPN),11:07,871722,Teio Sho (Local (Dirt),Flat,5,12,integer,1,integer,0.00,0.00,Kings Sword (JPN),Yusuke Fujioka,Ryo Terashima,,
8,1528614,2024-06-26,Ohi (JPN),11:07,871722,Teio Sho (Local (Dirt),Flat,5,8,integer,2,integer,1.75,1.75,Wilson Tesoro (JPN),Yuga Kawada,Hitoshi Kotegawa,,
9,1528615,2024-06-26,Ohi (JPN),11:07,871722,Teio Sho (Local (Dirt),Flat,5,3,integer,3,integer,2.75,1.00,Diktaean (JPN),Kazuo Yokoyama,Tatsuya Yoshioka,,


### Surviving-row pattern

The five exceptional races do not contain arbitrary internal gaps.

In every case, the surviving rows form a continuous sequence of numeric
finishing positions beginning at 1 and ending at the number of stored rows:

- seven stored rows contain positions 1–7 where `ran` is 8;
- four stored rows contain positions 1–4 where `ran` is 5;
- four stored rows contain positions 1–4 where `ran` is 5;
- two stored rows contain positions 1–2 where `ran` is 6; and
- fifteen stored rows contain positions 1–15 where `ran` is 16.

No surviving row records a fall, pull-up, disqualification or other textual
outcome.

The missing records therefore appear after the last stored finishing position.
This is consistent with result truncation or selective source coverage, but it
does not by itself distinguish between those explanations.

The runner-number values also demonstrate that `num` and `ran` represent
different concepts. Positive `num` values can exceed `ran`, and their sequences
need not begin at 1 or be contiguous.

In [7]:
# Summarise the position coverage of each race below `ran`.
#
# This confirms whether the visible pattern is genuinely a contiguous sequence
# from first place to the final stored row. We inspect storage classes as well
# as numeric minima and maxima so that textual outcomes cannot be accidentally
# treated as missing numeric positions.

below_ran_position_profile = pd.read_sql_query(
    """
    WITH affected_races AS (
        SELECT
            date,
            course,
            off
        FROM data
        WHERE rowid <> 1
        GROUP BY
            date,
            course,
            off
        HAVING
            COUNT(DISTINCT ran) = 1
            AND COUNT(*) < MAX(ran)
    )
    SELECT
        d.date,
        d.course,
        d.off,
        MAX(d.ran) AS reported_ran,
        COUNT(*) AS source_runner_rows,

        SUM(typeof(d.pos) = 'integer') AS integer_position_rows,
        SUM(typeof(d.pos) = 'text') AS text_position_rows,
        SUM(d.pos IS NULL) AS null_position_rows,

        MIN(
            CASE
                WHEN typeof(d.pos) = 'integer' THEN d.pos
            END
        ) AS minimum_numeric_position,

        MAX(
            CASE
                WHEN typeof(d.pos) = 'integer' THEN d.pos
            END
        ) AS maximum_numeric_position,

        COUNT(
            DISTINCT CASE
                WHEN typeof(d.pos) = 'integer' THEN d.pos
            END
        ) AS distinct_numeric_positions
    FROM data AS d
    INNER JOIN affected_races AS a
        ON d.date = a.date
        AND d.course = a.course
        AND d.off = a.off
    WHERE d.rowid <> 1
    GROUP BY
        d.date,
        d.course,
        d.off
    ORDER BY
        d.date,
        d.course,
        d.off
    """,
    connection,
)

# Derive an explicit sequence status rather than relying on visual inspection.
#
# A race is classified as a contiguous leading sequence only when:
#
# - every surviving row has an integer position;
# - the minimum position is 1;
# - the maximum position equals the stored row count; and
# - the number of distinct positions equals the stored row count.

below_ran_position_profile["position_sequence_status"] = (
    below_ran_position_profile.apply(
        lambda row: (
            "contiguous_leading_finishers"
            if (
                row["integer_position_rows"] == row["source_runner_rows"]
                and row["minimum_numeric_position"] == 1
                and row["maximum_numeric_position"]
                == row["source_runner_rows"]
                and row["distinct_numeric_positions"]
                == row["source_runner_rows"]
            )
            else "other_position_pattern"
        ),
        axis=1,
    )
)

display(below_ran_position_profile)

,date,course,off,reported_ran,source_runner_rows,integer_position_rows,text_position_rows,null_position_rows,minimum_numeric_position,maximum_numeric_position,distinct_numeric_positions,position_sequence_status
0,2024-06-18,Nantes (FR),2:14,8,7,7,0,0,1,7,7,contiguous_leading_finishers
1,2024-06-26,Ohi (JPN),11:07,5,4,4,0,0,1,4,4,contiguous_leading_finishers
2,2024-09-03,Morioka (JPN),11:07,5,4,4,0,0,1,4,4,contiguous_leading_finishers
3,2024-09-26,Funabashi (JPN),11:07,6,2,2,0,0,1,2,2,contiguous_leading_finishers
4,2025-10-09,Ohi (JPN),11:07,16,15,15,0,0,1,15,15,contiguous_leading_finishers


### External verification of the five exceptions

Because only five provisional races have fewer source rows than `ran`, each was
checked against a published result.

The exceptions do not share one universal explanation.

| Date | Course | Source `ran` | Stored rows | Published runners | External finding |
|---|---|---:|---:|---:|---|
| 2024-06-18 | Nantes (FR) | 8 | 7 | 8 | The missing starter, Saucats, fell. The seven stored rows are the seven finishers. |
| 2024-06-26 | Ohi (JPN) | 5 | 4 | 13 | The source stores only positions 1–4 and understates the actual field by eight runners. |
| 2024-09-03 | Morioka (JPN) | 5 | 4 | 12 | The source stores only positions 1–4 and understates the actual field by seven runners. |
| 2024-09-26 | Funabashi (JPN) | 6 | 2 | 6 | `ran` matches the published field, but positions 3–6 are absent. |
| 2025-10-09 | Ohi (JPN) | 16 | 15 | 16 | The missing runner, Tosen Thunder, failed to finish. |

This changes the interpretation of the source-wide equality result.

The agreement between source row count and `ran` in 189,038 races establishes
strong **internal consistency**, but it does not independently establish
complete runner coverage or prove that `ran` always means actual starters.

At least two races contain a repeated `ran` value that is demonstrably lower
than the published number of runners. Other races can therefore have equal
source row counts and `ran` while both omit part of the actual field.

Safe provisional interpretation:

- preserve `ran` as the source-presented race count;
- do not rename it `starter_count` without independent validation;
- distinguish internal row-count agreement from externally verified
  completeness;
- treat `source_rows = ran` as a consistency check, not proof of full coverage;
- retain explicit anomaly status for known externally contradicted values.

## Stage 3 — Search for concealed partial-field races

The five visible exceptions show that internal agreement between source row
count and `ran` is not sufficient evidence of complete race coverage.

Two Japanese races are particularly important:

- their stored rows contain only the leading finishers;
- `ran` understates the externally published field size; and
- retained positive runner numbers exceed `ran`.

A similar race could be concealed among the 189,038 cases where source row
count equals `ran`: the source could store a truncated subset and repeat the
subset size in `ran`.

This stage searches for internally complete-looking races with warning signals.

The first warning pattern is:

- source row count equals `ran`;
- all stored positions are distinct positive integers;
- the positions form a complete sequence from 1 through `ran`;
- at least one positive `num` exceeds `ran`.

This is a candidate-warning rule, not proof of an incomplete field. Runner
numbers can exceed the final field size after withdrawals or other numbering
conventions. Any resulting group must be profiled by jurisdiction and inspected
before it is classified.

In [8]:
# Build one record per provisional race and look for races that appear complete
# according to `ran` and finishing position, but whose runner numbers suggest
# that the stored rows may be only a subset of a larger numbered field.
#
# This is deliberately a warning screen rather than a definitive validator.
# Card numbers need not be contiguous and may exceed the number of starters
# after withdrawals. The purpose is to find concentrations and patterns that
# deserve closer investigation.

concealed_partial_candidates = pd.read_sql_query(
    """
    WITH race_profile AS (
        SELECT
            date,
            course,
            off,
            MIN(race_id) AS race_id,
            MIN(race_name) AS race_name,
            MIN(type) AS race_type,

            COUNT(*) AS source_runner_rows,
            MIN(ran) AS minimum_ran,
            MAX(ran) AS maximum_ran,
            COUNT(DISTINCT ran) AS distinct_ran_values,

            SUM(typeof(pos) = 'integer' AND pos > 0)
                AS positive_integer_position_rows,

            COUNT(
                DISTINCT CASE
                    WHEN typeof(pos) = 'integer' AND pos > 0
                    THEN pos
                END
            ) AS distinct_positive_positions,

            MIN(
                CASE
                    WHEN typeof(pos) = 'integer' AND pos > 0
                    THEN pos
                END
            ) AS minimum_positive_position,

            MAX(
                CASE
                    WHEN typeof(pos) = 'integer' AND pos > 0
                    THEN pos
                END
            ) AS maximum_positive_position,

            SUM(typeof(num) = 'integer' AND num > 0)
                AS positive_num_rows,

            MIN(
                CASE
                    WHEN typeof(num) = 'integer' AND num > 0
                    THEN num
                END
            ) AS minimum_positive_num,

            MAX(
                CASE
                    WHEN typeof(num) = 'integer' AND num > 0
                    THEN num
                END
            ) AS maximum_positive_num

        FROM data
        WHERE rowid <> 1
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        date,
        course,
        off,
        race_id,
        race_name,
        race_type,
        maximum_ran AS reported_ran,
        source_runner_rows,
        positive_num_rows,
        minimum_positive_num,
        maximum_positive_num,
        maximum_positive_num - maximum_ran AS num_above_ran
    FROM race_profile
    WHERE
        distinct_ran_values = 1
        AND source_runner_rows = maximum_ran

        -- Every stored row is a positive numeric finisher.
        AND positive_integer_position_rows = source_runner_rows
        AND distinct_positive_positions = source_runner_rows
        AND minimum_positive_position = 1
        AND maximum_positive_position = source_runner_rows

        -- At least one usable runner number exists and exceeds `ran`.
        AND positive_num_rows > 0
        AND maximum_positive_num > maximum_ran
    ORDER BY
        maximum_positive_num - maximum_ran DESC,
        date,
        course,
        off
    """,
    connection,
)

# Summarise the size of the candidate set before inspecting individual races.

concealed_partial_summary = pd.DataFrame(
    {
        "measure": [
            "candidate provisional races",
            "candidate source runner rows",
            "distinct courses",
            "first candidate date",
            "last candidate date",
            "largest num above ran",
        ],
        "value": [
            len(concealed_partial_candidates),
            int(concealed_partial_candidates["source_runner_rows"].sum()),
            concealed_partial_candidates["course"].nunique(),
            concealed_partial_candidates["date"].min(),
            concealed_partial_candidates["date"].max(),
            concealed_partial_candidates["num_above_ran"].max(),
        ],
    }
)

display(concealed_partial_summary)
display(concealed_partial_candidates.head(50))

,measure,value
0,candidate provisional races,58336
1,candidate source runner rows,562753
2,distinct courses,423
3,first candidate date,2015-01-01
4,last candidate date,2026-05-27
5,largest num above ran,13


,date,course,off,race_id,race_name,race_type,reported_ran,source_runner_rows,positive_num_rows,minimum_positive_num,maximum_positive_num,num_above_ran
0,2015-08-27,Clairefontaine (FR),12:50,642989,Prix Courezauxcourses.com (Prix De La Ville De Tourgeville) (Hurdle) (Handicap) (4yo) (Turf),Hurdle,3,3,3,1,16,13
1,2016-09-16,Newbury,5:30,657634,Bathwick Tyres Handicap,Flat,7,7,7,1,20,13
2,2022-07-31,Galway (IRE),2:00,818251,Adare Manor Opportunity Handicap Hurdle,Hurdle,8,8,8,1,21,13
3,2024-09-21,Newbury,5:00,874930,Conundrum Consulting Handicap,Flat,7,7,7,5,20,13
4,2021-09-25,Newmarket,3:40,790121,bet365 Cambridgeshire Handicap (Heritage Handicap),Flat,26,26,26,1,37,11
5,2022-09-07,Cork (IRE),5:45,820868,Follow Us On Instagram Handicap,Flat,16,16,16,1,26,10
6,2022-12-10,Naples (ITY),2:00,828852,U.N.I.R.E. Trofeo El Kabeir () (3yo+) (Turf),Flat,4,4,4,4,14,10
7,2025-06-02,Gowran Park (IRE),3:52,896886,QuinnBet Handicap,Flat,8,8,8,2,18,10
8,2015-07-25,Newmarket (July),3:40,630444,Adnams Broadside Handicap,Flat,11,11,11,2,20,9
9,2016-05-11,Bath,6:00,648840,Heros Charity Handicap,Flat,7,7,7,2,16,9


### Warning screen rejected

The proposed warning rule produced 58,336 races across 423 courses.

This is not a meaningful partial-field detector.

Runner numbers are card identifiers, not a sequence bounded by the final number
of starters. Withdrawals and non-runners routinely leave gaps, so a race with
`ran = 7` can legitimately retain runner numbers considerably above 7.

The output includes ordinary British, Irish, Australian, American and French
races where this explanation is sufficient. Consequently:

- `maximum num > ran` must not be treated as evidence of missing source rows;
- the candidate set must not be persisted as an anomaly table;
- this rule must not appear in a validator; and
- runner-number magnitude cannot independently establish field completeness.

The exercise nevertheless confirms an important semantic point: `num` is not an
ordinal position within the final field and cannot be validated against `ran`
using a simple upper bound.

The two externally contradicted Japanese races demonstrate that concealed
partial fields can exist, but the current source fields do not provide a
universal internal method for detecting them. Coverage equality must therefore
remain an internal-consistency measure rather than proof of external
completeness.

In [9]:
# Identify repeated positive runner numbers within provisional races.
#
# Blank text and integer zero are excluded here because they are already known
# missing/sentinel states. This query tests only whether a positive source
# number uniquely identifies one stored runner inside a race.

duplicate_positive_num_groups = pd.read_sql_query(
    """
    SELECT
        date,
        course,
        off,
        num,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT horse) AS distinct_horses,
        MIN(race_id) AS minimum_race_id,
        MAX(race_id) AS maximum_race_id,
        MIN(race_name) AS race_name,
        MIN(type) AS race_type
    FROM data
    WHERE
        rowid <> 1
        AND typeof(num) = 'integer'
        AND num > 0
    GROUP BY
        date,
        course,
        off,
        num
    HAVING COUNT(*) > 1
    ORDER BY
        runner_rows DESC,
        date,
        course,
        off,
        num
    """,
    connection,
)

# Summarise the scale of positive-number duplication before examining examples.
#
# Distinct horses are retained because repeated rows for the same horse would
# indicate a different source problem from one number being assigned to
# multiple runners.

duplicate_positive_num_summary = pd.DataFrame(
    {
        "measure": [
            "duplicate positive-num groups",
            "affected provisional races",
            "affected runner rows",
            "groups containing multiple horses",
            "maximum rows sharing one positive num",
            "first affected date",
            "last affected date",
        ],
        "value": [
            len(duplicate_positive_num_groups),
            duplicate_positive_num_groups[
                ["date", "course", "off"]
            ].drop_duplicates().shape[0],
            int(duplicate_positive_num_groups["runner_rows"].sum()),
            int(
                (
                    duplicate_positive_num_groups["distinct_horses"] > 1
                ).sum()
            ),
            int(duplicate_positive_num_groups["runner_rows"].max()),
            duplicate_positive_num_groups["date"].min(),
            duplicate_positive_num_groups["date"].max(),
        ],
    }
)

display(duplicate_positive_num_summary)
display(duplicate_positive_num_groups.head(50))

,measure,value
0,duplicate positive-num groups,523
1,affected provisional races,362
2,affected runner rows,1084
3,groups containing multiple horses,523
4,maximum rows sharing one positive num,4
5,first affected date,2015-01-01
6,last affected date,2026-04-02


,date,course,off,num,runner_rows,distinct_horses,minimum_race_id,maximum_race_id,race_name,race_type
0,2018-12-08,San Isidro (ARG),9:55,11,4,4,718465,718465,Copa de Plata (3yo+ Fillies & Mares) (Turf),Flat
1,2015-01-11,Monterrico (PER),10:25,10,3,3,617495,617495,Premio Enrique Meiggs (3yo+) (Dirt),Flat
2,2015-03-07,Aqueduct (USA),9:50,1,3,3,620968,620968,Gotham Stakes (3yo) (Dirt),Flat
3,2015-07-05,Club Hipico de Santiago (CHI),8:35,2,3,3,632844,632844,Premio Arturo Lyon P (3yo Fillies) (Turf),Flat
4,2015-08-02,Club Hipico de Santiago (CHI),8:35,3,3,3,632852,632852,Premio Polla de Potrancas (3yo Fillies) (Turf),Flat
5,2015-09-06,Monterrico (PER),10:50,3,3,3,634807,634807,Premio Polla de Potrillos - Roberto Alvarez Calderon Rey (3yo Colts & Geldings) (Dirt),Flat
6,2015-10-04,Monterrico (PER),10:55,2,3,3,636568,636568,Ricardo Ortiz de Zevallos (3yo) (Dirt),Flat
7,2015-11-08,Monterrico (PER),10:35,9,3,3,638970,638970,Derby Nacional (3yo) (Dirt),Flat
8,2015-11-08,Monterrico (PER),10:35,10,3,3,638970,638970,Derby Nacional (3yo) (Dirt),Flat
9,2015-11-14,San Isidro (ARG),8:25,5,3,3,639388,639388,Premio Enrique Acebal (3yo Fillies) (Turf),Flat


## Stage 4 — Duplicate positive runner numbers

Positive runner numbers are not universally unique within provisional races.

The source contains:

- 523 race-and-number groups containing more than one runner row;
- 362 affected provisional races;
- 1,084 affected runner rows;
- different horses in every duplicated group; and
- up to four horses sharing one positive `num`.

The affected races are concentrated in jurisdictions where coupled or bracketed
betting entries are common, including the United States and several South
American jurisdictions.

This is strong evidence that `num` may identify an entry or betting interest
rather than one individual horse in some jurisdictions.

The integer field does not preserve visible suffixes such as `1A`, `1B` or
similar distinctions. The next step inspects complete runner rows from selected
duplicate groups to determine whether those distinctions survive in another
field or have been flattened during source construction.

In [10]:
# Select a small set of informative duplicate-number examples.
#
# These include:
#
# - the only observed four-runner group;
# - a three-runner United States example;
# - a two-runner United States example; and
# - South American examples from several jurisdictions.
#
# The selection is deliberately small because the aim is to understand the
# representation, not to print all 1,084 affected rows.

selected_duplicate_groups = pd.DataFrame(
    [
        {
            "date": "2018-12-08",
            "course": "San Isidro (ARG)",
            "off": "9:55",
            "num": 11,
        },
        {
            "date": "2015-03-07",
            "course": "Aqueduct (USA)",
            "off": "9:50",
            "num": 1,
        },
        {
            "date": "2015-01-01",
            "course": "Aqueduct (USA)",
            "off": "6:20",
            "num": 1,
        },
        {
            "date": "2015-01-20",
            "course": "Gavea (BRZ)",
            "off": "8:15",
            "num": 4,
        },
        {
            "date": "2015-01-11",
            "course": "Monterrico (PER)",
            "off": "10:25",
            "num": 10,
        },
    ]
)

# Load all rows for the selected races rather than only the duplicated number.
#
# Seeing the surrounding runner-number sequence helps distinguish:
#
# - genuine coupled entries;
# - broad duplication affecting the entire race;
# - source-row duplication; and
# - unrelated corruption.
#
# Fields such as horse, draw, position, price, owner and comment are included
# because a suffix or coupling indicator might have survived outside `num`.

selected_duplicate_rows = pd.read_sql_query(
    """
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type AS race_type,
        ran,
        num,
        draw,
        pos,
        horse,
        jockey,
        trainer,
        owner,
        sp,
        comment
    FROM data
    WHERE
        rowid <> 1
        AND (
            (date = '2018-12-08'
             AND course = 'San Isidro (ARG)'
             AND off = '9:55')
            OR
            (date = '2015-03-07'
             AND course = 'Aqueduct (USA)'
             AND off = '9:50')
            OR
            (date = '2015-01-01'
             AND course = 'Aqueduct (USA)'
             AND off = '6:20')
            OR
            (date = '2015-01-20'
             AND course = 'Gavea (BRZ)'
             AND off = '8:15')
            OR
            (date = '2015-01-11'
             AND course = 'Monterrico (PER)'
             AND off = '10:25')
        )
    ORDER BY
        date,
        course,
        off,
        CASE
            WHEN typeof(num) = 'integer' THEN num
            ELSE 999
        END,
        CASE
            WHEN typeof(pos) = 'integer' THEN pos
            ELSE 999
        END,
        rowid
    """,
    connection,
)

# Mark the specifically duplicated number within each selected race.
#
# This makes the target groups easy to identify while retaining all surrounding
# runners for context.

selected_duplicate_rows = selected_duplicate_rows.merge(
    selected_duplicate_groups.assign(selected_duplicate_num=True),
    on=["date", "course", "off", "num"],
    how="left",
)

selected_duplicate_rows["selected_duplicate_num"] = (
    selected_duplicate_rows["selected_duplicate_num"]
    .fillna(False)
    .astype(bool)
)

display(selected_duplicate_rows)

,source_rowid,date,course,off,race_id,race_name,race_type,ran,num,draw,pos,horse,jockey,trainer,owner,sp,comment,selected_duplicate_num
0,54,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,1,1,6,Simply Spectacular (USA),Ruben Silvera,Michael Wilson,Edward A Seltzer Beverly S Anderson,31/1,,True
1,143,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,1,4,8,Mylitta (USA),Angel S Arroyo,Michael Wilson,Edward A Seltzer,31/1,,True
2,50,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,2,3,2,Penwith (USA),Fernando Jara,Kiaran McLaughlin,Godolphin Racing Llc,4/1,Soon led - ridden 2f out - quickened and 3 lengths clear entering final furlong - flashed tail under pressure - look...,False
3,51,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,2,7,3,Divided Attention (USA),Dylan Davis,Kiaran McLaughlin,Godolphin Racing Llc,4/1,Held up - ridden into straight - kept on steadily and went 3rd towards finish - not pace to challenge,False
4,52,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,3,2,4,Shayjolie (USA),Jose L Ortiz,Gary Contessa,Sean Shay,13/2,,False
5,49,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,4,5,1,America (USA),Junior Alvarado,William Mott,Bobby Flay,42/10,,False
6,55,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,5,6,7,Agawa (CAN),Charles C Lopez,Chris Englehart,Wachtel Stable,40/1,,False
7,53,2015-01-01,Aqueduct (USA),6:20,616883,Affectionately Stakes () (4yo+ Fillies & Mares) (Dirt),Flat,8,7,8,5,Belle Gallantey (USA),Irad Ortiz Jr,Rudy Rodriguez,Michael Dubb Bethlehem Stables Llc,3/5F,,False
8,3756,2015-01-11,Monterrico (PER),10:25,617495,Premio Enrique Meiggs (3yo+) (Dirt),Flat,12,1,1,9,Dr. Rodrigo (ARG),A Siguas,F Aburto,Op Stables,,,False
9,3749,2015-01-11,Monterrico (PER),10:25,617495,Premio Enrique Meiggs (3yo+) (Dirt),Flat,12,2,2,1,Liberal (PER),Edwin R Talaverano,C Traverso II,The Fathers,23/10,,False


### Coupled-entry representation confirmed

Inspection of complete runner rows confirms that duplicated positive `num`
values are generally coupled or bracketed betting entries rather than duplicate
runner records.

Horses sharing one `num` frequently also share:

- the same starting price;
- the same owner;
- the same trainer; or
- a combination of those attributes.

Examples include:

- two Aqueduct runners sharing `num = 1` and SP `31/1`;
- two Aqueduct runners sharing `num = 2` and SP `4/1`;
- three Gotham Stakes runners sharing `num = 1` and SP `69/20`;
- three Monterrico runners sharing `num = 10`, trainer and owner;
- paired Brazilian runners sharing number, owner and price; and
- four San Isidro runners sharing `num = 11`, owner and SP `162/10`.

The source integer field has therefore flattened any original suffix or
sub-entry distinction such as `1A`, `1B` or equivalent.

Governed conclusion:

- `num` may identify a betting interest rather than one horse;
- positive `num` is not universally unique within a race;
- `race identity + num` must not be used as a runner key;
- duplicate positive numbers must not automatically be classified as data
  duplication;
- the raw integer value should be preserved;
- no missing suffix should be reconstructed without an authoritative source.

## Stage 5 — Jurisdictional distribution of duplicate numbers

The inspected examples show that duplicated positive `num` values can represent
legitimate coupled betting entries.

That does not mean every duplicated number should be accepted without context.
The convention may be normal in some jurisdictions and exceptional in others.

This stage will therefore profile duplicate positive-number groups by governed
course jurisdiction and year.

Course jurisdiction will be taken from the durable course-location reference
created by Notebook 12. It will not be re-derived independently from course
names inside this notebook.

In [11]:
# Load the governed course-location reference created by Notebook 12.
#
# Notebook 14 should reuse this durable mapping rather than introduce another
# course-to-jurisdiction derivation. We first inspect its columns so that the
# subsequent join uses the documented reference structure correctly.

COURSE_LOCATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "course_locations.csv"
)

assert COURSE_LOCATIONS_PATH.exists(), (
    f"Course-location reference not found: {COURSE_LOCATIONS_PATH}"
)

course_locations = pd.read_csv(COURSE_LOCATIONS_PATH)

print(f"Reference rows: {len(course_locations):,}")
display(course_locations.head())
display(
    pd.DataFrame(
        {
            "column": course_locations.columns,
            "dtype": [
                str(course_locations[column].dtype)
                for column in course_locations.columns
            ],
        }
    )
)

Reference rows: 395


,candidate_course_label,candidate_jurisdiction,physical_venue_name,locality,region,country,latitude,longitude,iana_timezone,location_evidence,location_validation_status,raw_course_labels,provisional_races,meeting_dates,earliest_date,latest_date
0,La Plata,Argentina,Hipódromo de La Plata,La Plata,Buenos Aires,Argentina,-34.901267,-57.943804,America/Argentina/Buenos_Aires,"Nominatim manual selection from query 'Hipódromo de La Plata, Argentina'; provider place way:287312802; raw response...",manually_validated,La Plata (ARG),37,28,2015-01-18,2025-09-21
1,Palermo,Argentina,Hipódromo Argentino de Palermo,Buenos Aires,Autonomous City of Buenos Aires,Argentina,-34.566398,-58.425727,America/Argentina/Buenos_Aires,"Nominatim manual selection from query 'Palermo hipódromo, Argentina'; selected result index 0, Hipódromo Argentino d...",manually_validated,Palermo (ARG),200,76,2015-01-03,2025-10-11
2,San Isidro,Argentina,NaN,NaN,NaN,NaN,NaN,NaN,America/Argentina/Buenos_Aires,NaN,unassigned,San Isidro (ARG),179,80,2015-02-07,2025-10-04
3,Albury,Australia,NaN,NaN,NaN,NaN,NaN,NaN,Australia/Sydney,NaN,unassigned,Albury (AUS),4,1,2015-03-27,2015-03-27
4,Alice springs,Australia,NaN,NaN,NaN,NaN,NaN,NaN,Australia/Darwin,NaN,unassigned,Alice springs (AUS),1,1,2019-05-06,2019-05-06


,column,dtype
0,candidate_course_label,str
1,candidate_jurisdiction,str
2,physical_venue_name,str
3,locality,str
4,region,str
5,country,str
6,latitude,float64
7,longitude,float64
8,iana_timezone,str
9,location_evidence,str


### Governed jurisdiction reference

The Notebook 12 reference contains one governed row for each source course identity.

For this investigation:

- source `course` joins to `raw_course_labels`;
- jurisdiction is taken from `candidate_jurisdiction`;
- no new jurisdiction parsing is introduced.

The reference contains 395 course identities, matching the current durable baseline.

In [12]:
# Attach governed jurisdiction to every duplicated positive-number group.
#
# Many duplicate groups can occur at one course, but each raw course label
# should map to exactly one governed jurisdiction.

duplicate_positive_num_with_jurisdiction = (
    duplicate_positive_num_groups
    .merge(
        course_locations[
            [
                "raw_course_labels",
                "candidate_jurisdiction",
            ]
        ],
        left_on="course",
        right_on="raw_course_labels",
        how="left",
        validate="many_to_one",
    )
)

# Confirm whether any affected course failed to join to the governed reference.

unresolved_duplicate_group_courses = (
    duplicate_positive_num_with_jurisdiction.loc[
        duplicate_positive_num_with_jurisdiction[
            "candidate_jurisdiction"
        ].isna(),
        ["course"],
    ]
    .drop_duplicates()
    .sort_values("course")
    .reset_index(drop=True)
)

print(
    "Unresolved affected course labels:",
    len(unresolved_duplicate_group_courses),
)

if not unresolved_duplicate_group_courses.empty:
    display(unresolved_duplicate_group_courses)

# Keep only duplicated-number groups outside the six jurisdictions where the
# convention is already common and persistent.

common_coupled_entry_jurisdictions = {
    "Argentina",
    "Brazil",
    "Chile",
    "Peru",
    "United States",
    "Uruguay",
}

rare_duplicate_num_groups = (
    duplicate_positive_num_with_jurisdiction.loc[
        (
            ~duplicate_positive_num_with_jurisdiction[
                "candidate_jurisdiction"
            ].isin(common_coupled_entry_jurisdictions)
        )
        |
        (
            duplicate_positive_num_with_jurisdiction[
                "candidate_jurisdiction"
            ].isna()
        )
    ]
    .copy()
    .sort_values(
        [
            "candidate_jurisdiction",
            "date",
            "course",
            "off",
            "num",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

# Derive the affected provisional-race keys for the next SQLite query.

rare_race_keys = (
    rare_duplicate_num_groups[
        [
            "date",
            "course",
            "off",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(
    rare_duplicate_num_groups[
        [
            "date",
            "course",
            "off",
            "race_name",
            "race_type",
            "num",
            "runner_rows",
            "distinct_horses",
            "candidate_jurisdiction",
        ]
    ]
)

Unresolved affected course labels: 1


,course
0,Bahrain


,date,course,off,race_name,race_type,num,runner_rows,distinct_horses,candidate_jurisdiction
0,2015-03-19,Saint-Cloud (FR),2:55,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,4,2,2,France
1,2022-09-02,Auteuil (FR),11:20,Prix Weather Permitting (Chase) (Conditions) (3yo Unraced Over Fences) (Turf),Chase,3,2,2,France
2,2023-10-11,Chantilly (FR),12:25,Prix de Coye (Conditions) (4yo+) (All-Weather Track) (Polytrack),Flat,6,2,2,France
3,2024-01-05,Deauville (FR),5:58,Prix de lAigle (Handicap) (4yo+) (All-Weather Track) (Polytrack),Flat,9,2,2,France
4,2019-10-16,Happy Valley (HK),2:15,The American Club Challenge Cup (Handicap) (3yo+) (Course C) (Turf),Flat,10,2,2,Hong Kong
5,2023-12-08,Meydan (UAE),5:35,AZIZI REVE Presented by Azizi (Handicap) (Dirt),Flat,8,2,2,United Arab Emirates
6,2024-01-19,Meydan (UAE),5:35,Dubai Auto Zone Presented by DP World (Handicap) (Dirt),Flat,3,2,2,United Arab Emirates
7,2026-04-02,Bahrain,17:00,Coolmore Cup (Handicap) (Turf),Flat,5,2,2,NaN


In [13]:
# Recreate the small set of affected provisional-race keys.
#
# The kernel restart cleared all in-memory variables, so this cell derives the
# keys again from `rare_duplicate_num_groups` before querying SQLite.

rare_race_keys = (
    rare_duplicate_num_groups[
        [
            "date",
            "course",
            "off",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

rare_race_key_records = list(
    rare_race_keys.itertuples(index=False, name=None)
)

connection.execute(
    """
    CREATE TEMP TABLE IF NOT EXISTS rare_race_keys_temp (
        date TEXT NOT NULL,
        course TEXT NOT NULL,
        off TEXT NOT NULL,
        PRIMARY KEY (date, course, off)
    )
    """
)

connection.execute("DELETE FROM rare_race_keys_temp")

connection.executemany(
    """
    INSERT INTO rare_race_keys_temp (
        date,
        course,
        off
    )
    VALUES (?, ?, ?)
    """,
    rare_race_key_records,
)

rare_duplicate_race_rows = pd.read_sql_query(
    """
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.off,
        d.race_id,
        d.race_name,
        d.type AS race_type,
        d.ran,
        d.num,
        d.draw,
        d.pos,
        d.horse,
        d.jockey,
        d.trainer,
        d.owner,
        d.sp,
        d.comment
    FROM data AS d
    INNER JOIN rare_race_keys_temp AS k
        ON d.date = k.date
        AND d.course = k.course
        AND d.off = k.off
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        CASE
            WHEN typeof(d.num) = 'integer' THEN d.num
            ELSE 999
        END,
        CASE
            WHEN typeof(d.pos) = 'integer' THEN d.pos
            ELSE 999
        END,
        d.rowid
    """,
    connection,
)

rare_duplicate_race_rows = (
    rare_duplicate_race_rows
    .merge(
        rare_duplicate_num_groups[
            [
                "date",
                "course",
                "off",
                "num",
                "candidate_jurisdiction",
            ]
        ].assign(duplicated_num=True),
        on=["date", "course", "off", "num"],
        how="left",
        validate="many_to_one",
    )
)

rare_duplicate_race_rows["duplicated_num"] = (
    rare_duplicate_race_rows["duplicated_num"]
    .fillna(False)
    .astype(bool)
)

display(rare_duplicate_race_rows)

,source_rowid,date,course,off,race_id,race_name,race_type,ran,num,draw,pos,horse,jockey,trainer,owner,sp,comment,candidate_jurisdiction,duplicated_num
0,26730,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,1,5,4,Frantz De Galais (FR),Nicolas Larenaudie,J-M Lefebvre,Mme Caroline Petit,26/5,,NaN,False
1,26732,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,3,8,2,Quid Flight (FR),Ronan Thomas,J Phelippon,Mme C Dufaut J Phelippon,6/5F,,NaN,False
2,26728,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,4,7,6,Atomic Bere (FR),Frank Panicucci,Mlle C Cardenne,R J Philippe Mlle C Cardenne E Devise,84/10,,France,True
3,26724,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,4,4,10,Dence (FR),David Breux,Mlle C Comte,Mlle Celine Comte,53/1,,France,True
4,26722,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,5,9,12,Amuse Gueule (GER),Frederic Spanu,J-P Perruchot,Maurice Bertin,27/1,,NaN,False
5,26726,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,7,7,8,Kissavos (GB),Morgan Delalande,Y Barberot,David Bourillon,117/10,,NaN,False
6,26725,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,8,12,9,Mobi (FR),Pierre-Charles Boudot,J Phelippon,F Guy J Gerard J Phelippon,13/1,,NaN,False
7,26731,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,9,11,3,Teo Le Sarde (FR),Alexis Badel,A Junk,L Chauvin A Junk,96/10,,NaN,False
8,26694,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,10,2,13,Passarinho (IRE),Fabrice Veron,H-A Pantall,Alexandre Pereira,139/10,,NaN,False
9,26733,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,11,1,1,Decoy (FR),Tony Piccone,Mme A Rosa,Mlle Alexandra Rosa,231/10,,NaN,False


### Jurisdictional finding

Duplicate positive runner numbers are overwhelmingly concentrated in a small
set of jurisdictions.

Of the 523 duplicated race-and-number groups:

- 184 occur in Argentina;
- 167 occur in Brazil;
- 71 occur in Peru;
- 53 occur in the United States;
- 34 occur in Uruguay; and
- 6 occur in Chile.

These six jurisdictions account for 515 of the 523 groups.

The convention also recurs across multiple years within each major affected
jurisdiction. This supports the interpretation that duplicated positive
numbers usually represent legitimate coupled or bracketed betting entries
rather than accidental duplicate source records.

Eight groups remain outside that dominant pattern:

- four in France;
- two in the United Arab Emirates;
- one in Hong Kong; and
- one at the unresolved source course label `Bahrain`.

These rare cases require direct inspection before they can be classified.

The Bahrain row also failed to join to the governed Notebook 12 course
reference. This must be investigated separately rather than silently assigned
a jurisdiction from the course text.

In [14]:
# Retrieve only the runner rows belonging to the rare affected races.
#
# The previous version loaded the entire 1.85-million-row source table into
# pandas and filtered afterwards. That was wasteful and could exhaust the
# notebook kernel's memory.
#
# This version constructs a small in-memory table of affected provisional-race
# keys inside SQLite, then joins the source directly to those keys. Only the
# required rows are returned to pandas.

rare_race_key_records = list(
    rare_race_keys[
        [
            "date",
            "course",
            "off",
        ]
    ].itertuples(index=False, name=None)
)

connection.execute(
    """
    CREATE TEMP TABLE IF NOT EXISTS rare_race_keys_temp (
        date TEXT NOT NULL,
        course TEXT NOT NULL,
        off TEXT NOT NULL,
        PRIMARY KEY (date, course, off)
    )
    """
)

# Clear any rows left by an earlier execution of this cell.
connection.execute("DELETE FROM rare_race_keys_temp")

connection.executemany(
    """
    INSERT INTO rare_race_keys_temp (
        date,
        course,
        off
    )
    VALUES (?, ?, ?)
    """,
    rare_race_key_records,
)

rare_duplicate_race_rows = pd.read_sql_query(
    """
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.off,
        d.race_id,
        d.race_name,
        d.type AS race_type,
        d.ran,
        d.num,
        d.draw,
        d.pos,
        d.horse,
        d.jockey,
        d.trainer,
        d.owner,
        d.sp,
        d.comment
    FROM data AS d
    INNER JOIN rare_race_keys_temp AS k
        ON d.date = k.date
        AND d.course = k.course
        AND d.off = k.off
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        CASE
            WHEN typeof(d.num) = 'integer' THEN d.num
            ELSE 999
        END,
        CASE
            WHEN typeof(d.pos) = 'integer' THEN d.pos
            ELSE 999
        END,
        d.rowid
    """,
    connection,
)

# Mark the specifically duplicated number within each selected race.
#
# Rows belonging to the same race but carrying other runner numbers remain in
# the output for context.

rare_duplicate_race_rows = (
    rare_duplicate_race_rows
    .merge(
        rare_duplicate_num_groups[
            [
                "date",
                "course",
                "off",
                "num",
                "candidate_jurisdiction",
            ]
        ].assign(duplicated_num=True),
        on=["date", "course", "off", "num"],
        how="left",
        validate="many_to_one",
    )
)

rare_duplicate_race_rows["duplicated_num"] = (
    rare_duplicate_race_rows["duplicated_num"]
    .fillna(False)
    .astype(bool)
)

display(rare_duplicate_race_rows)

,source_rowid,date,course,off,race_id,race_name,race_type,ran,num,draw,pos,horse,jockey,trainer,owner,sp,comment,candidate_jurisdiction,duplicated_num
0,26730,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,1,5,4,Frantz De Galais (FR),Nicolas Larenaudie,J-M Lefebvre,Mme Caroline Petit,26/5,,NaN,False
1,26732,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,3,8,2,Quid Flight (FR),Ronan Thomas,J Phelippon,Mme C Dufaut J Phelippon,6/5F,,NaN,False
2,26728,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,4,7,6,Atomic Bere (FR),Frank Panicucci,Mlle C Cardenne,R J Philippe Mlle C Cardenne E Devise,84/10,,France,True
3,26724,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,4,4,10,Dence (FR),David Breux,Mlle C Comte,Mlle Celine Comte,53/1,,France,True
4,26722,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,5,9,12,Amuse Gueule (GER),Frederic Spanu,J-P Perruchot,Maurice Bertin,27/1,,NaN,False
5,26726,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,7,7,8,Kissavos (GB),Morgan Delalande,Y Barberot,David Bourillon,117/10,,NaN,False
6,26725,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,8,12,9,Mobi (FR),Pierre-Charles Boudot,J Phelippon,F Guy J Gerard J Phelippon,13/1,,NaN,False
7,26731,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,9,11,3,Teo Le Sarde (FR),Alexis Badel,A Junk,L Chauvin A Junk,96/10,,NaN,False
8,26694,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,10,2,13,Passarinho (IRE),Fabrice Veron,H-A Pantall,Alexandre Pereira,139/10,,NaN,False
9,26733,2015-03-19,Saint-Cloud (FR),2:55,621971,Prix de Juvisy (Claimer) (4yo+) (Turf),Flat,13,11,1,1,Decoy (FR),Tony Piccone,Mme A Rosa,Mlle Alexandra Rosa,231/10,,NaN,False


### Rare duplicate-number cases

Individual inspection shows that duplicated positive `num` values outside the
main coupling jurisdictions do not all share one explanation.

Several French, Hong Kong and United Arab Emirates examples contain horses with
the same `num` but different:

- owners;
- trainers;
- starting prices; and
- draws.

These do not resemble the confirmed coupled-entry examples and are better
treated as ambiguous source-number collisions.

The Bahrain example is different. The two horses sharing `num = 5` also share
draw 1 and the same owner, which is consistent with a coupled or bracketed
entry. However, `Bahrain` did not join to the governed course-location
reference and therefore also exposes a separate reference-maintenance issue.

Governed conclusion:

- duplicated positive `num` is compatible with legitimate coupled entries;
- duplicated positive `num` does not by itself prove coupling;
- no universal uniqueness constraint is safe;
- no universal coupled-entry classification can be derived from `num` alone;
- duplicate-number groups outside known convention patterns should carry an
  ambiguous status unless independently verified;
- the Bahrain course identity requires reconciliation with the Notebook 12
  reference.

## Stage 6 — Blank and zero runner numbers

The raw profile identified two non-positive `num` states:

- 7,032 blank-text rows across 877 provisional races;
- 1,179 integer-zero rows across 186 provisional races.

These states must not be collapsed automatically.

A blank text value and an explicit integer zero may represent different source
behaviour, such as:

- a jurisdiction that does not supply runner numbers;
- a race where numbering was omitted;
- a source-system sentinel;
- a partial or degraded import;
- an unnumbered participant; or
- another jurisdiction-specific convention.

This stage profiles blank and zero values by governed jurisdiction, year and
race type.

The purpose is to determine whether either state can be assigned a narrower
governed meaning. Until then, both remain preserved source states rather than
being converted to one generic null value.

In [15]:
# Extract every runner row where `num` is either blank text or integer zero.
#
# The two states are labelled separately so that later summaries cannot merge
# them accidentally.

nonpositive_num_rows = pd.read_sql_query(
    """
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type AS race_type,
        ran,
        num,
        typeof(num) AS num_storage_class,
        pos,
        horse
    FROM data
    WHERE
        rowid <> 1
        AND (
            (
                typeof(num) = 'text'
                AND trim(CAST(num AS TEXT)) = ''
            )
            OR
            (
                typeof(num) = 'integer'
                AND num = 0
            )
        )
    """,
    connection,
)

# Assign an explicit raw-state label.
#
# No semantic name such as `missing`, `unknown` or `sentinel` is used yet,
# because the source meaning has not been established.

nonpositive_num_rows["num_raw_state"] = nonpositive_num_rows.apply(
    lambda row: (
        "blank_text"
        if row["num_storage_class"] == "text"
        else "integer_zero"
    ),
    axis=1,
)

# Attach the governed jurisdiction from the Notebook 12 course reference.
#
# A many-to-one validation is required because many runner rows can belong to
# one course identity, while each raw source course label should map to one
# governed jurisdiction.

nonpositive_num_rows = (
    nonpositive_num_rows
    .merge(
        course_locations[
            [
                "raw_course_labels",
                "candidate_jurisdiction",
            ]
        ],
        left_on="course",
        right_on="raw_course_labels",
        how="left",
        validate="many_to_one",
    )
)

# Extract source year for temporal profiling.

nonpositive_num_rows["year"] = (
    nonpositive_num_rows["date"]
    .astype(str)
    .str.slice(0, 4)
    .astype(int)
)

# Check whether any affected course labels failed to join to the governed
# reference before interpreting the jurisdictional summaries.

unresolved_nonpositive_num_courses = (
    nonpositive_num_rows.loc[
        nonpositive_num_rows["candidate_jurisdiction"].isna(),
        ["course"],
    ]
    .drop_duplicates()
    .sort_values("course")
    .reset_index(drop=True)
)

print(
    "Unresolved affected course labels:",
    len(unresolved_nonpositive_num_courses),
)

if not unresolved_nonpositive_num_courses.empty:
    display(unresolved_nonpositive_num_courses)

# Summarise blank and zero states by governed jurisdiction.
#
# We retain both runner-row counts and affected race counts because one race may
# contain several rows in the same state.

nonpositive_num_by_jurisdiction = (
    nonpositive_num_rows
    .groupby(
        [
            "num_raw_state",
            "candidate_jurisdiction",
        ],
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        affected_provisional_races=(
            "date",
            lambda values: (
                nonpositive_num_rows.loc[
                    values.index,
                    ["date", "course", "off"],
                ]
                .drop_duplicates()
                .shape[0]
            ),
        ),
        distinct_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
    .sort_values(
        [
            "num_raw_state",
            "runner_rows",
        ],
        ascending=[True, False],
    )
)

# Summarise by jurisdiction and year to reveal persistent conventions,
# historical changes or isolated source periods.

nonpositive_num_by_jurisdiction_year = (
    nonpositive_num_rows
    .groupby(
        [
            "num_raw_state",
            "candidate_jurisdiction",
            "year",
        ],
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        affected_provisional_races=(
            "date",
            lambda values: (
                nonpositive_num_rows.loc[
                    values.index,
                    ["date", "course", "off"],
                ]
                .drop_duplicates()
                .shape[0]
            ),
        ),
        distinct_courses=("course", "nunique"),
    )
    .reset_index()
    .sort_values(
        [
            "num_raw_state",
            "candidate_jurisdiction",
            "year",
        ]
    )
)

# Summarise by race type to determine whether either state is associated with
# a particular racing code rather than primarily with jurisdiction.

nonpositive_num_by_race_type = (
    nonpositive_num_rows
    .groupby(
        [
            "num_raw_state",
            "race_type",
        ],
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        affected_provisional_races=(
            "date",
            lambda values: (
                nonpositive_num_rows.loc[
                    values.index,
                    ["date", "course", "off"],
                ]
                .drop_duplicates()
                .shape[0]
            ),
        ),
        distinct_courses=("course", "nunique"),
    )
    .reset_index()
    .sort_values(
        [
            "num_raw_state",
            "runner_rows",
        ],
        ascending=[True, False],
    )
)

display(nonpositive_num_by_jurisdiction)
display(nonpositive_num_by_jurisdiction_year)
display(nonpositive_num_by_race_type)

Unresolved affected course labels: 7


,course
0,Bordeaux Le Bouscat
1,Chukyo
2,Cidade Jardim
3,Hipodromo Chile
4,Les Landes
5,Monterrico
6,Nakayama


,num_raw_state,candidate_jurisdiction,runner_rows,affected_provisional_races,distinct_courses,first_date,last_date
10,blank_text,Japan,1972,169,17,2015-01-04,2025-10-09
11,blank_text,Jersey,1196,193,1,2015-04-06,2025-08-25
21,blank_text,United States,1115,156,32,2015-01-02,2025-03-29
7,blank_text,Germany,651,88,14,2015-04-05,2025-07-05
1,blank_text,Australia,422,42,15,2015-01-01,2024-08-31
9,blank_text,Italy,360,62,6,2015-04-01,2025-06-08
6,blank_text,France,319,55,27,2015-02-07,2025-09-29
12,blank_text,New Zealand,228,18,7,2015-01-01,2015-02-28
18,blank_text,Sweden,176,26,3,2015-05-12,2021-10-17
14,blank_text,Qatar,138,10,1,2015-11-17,2020-02-01


,num_raw_state,candidate_jurisdiction,year,runner_rows,affected_provisional_races,distinct_courses
0,blank_text,Argentina,2015,4,1,1
1,blank_text,Australia,2015,354,33,12
2,blank_text,Australia,2016,7,1,1
3,blank_text,Australia,2017,3,1,1
4,blank_text,Australia,2019,5,1,1
...,...,...,...,...,...,...
129,integer_zero,United States,2018,6,1,1
130,integer_zero,United States,2019,1,1,1
131,integer_zero,United States,2022,1,1,1
132,integer_zero,United States,2025,20,5,4


,num_raw_state,race_type,runner_rows,affected_provisional_races,distinct_courses
1,blank_text,Flat,6715,814,136
2,blank_text,Hurdle,202,48,10
0,blank_text,Chase,115,15,10
4,integer_zero,Flat,1030,155,22
5,integer_zero,Hurdle,145,30,1
3,integer_zero,Chase,4,1,1


### Initial blank-and-zero finding

Blank text and integer zero have different source distributions.

Blank values occur across many jurisdictions and periods, with the largest
concentrations in Japan, Jersey, the United States, Germany, Australia and
Italy.

Integer zero is much more concentrated:

- 911 of the 1,179 zero rows occur in Jersey;
- Guernsey contributes a further 57 rows;
- the remaining zeroes occur in small jurisdictional pockets.

This makes it unsafe to collapse blank text and integer zero into one raw state
during ingestion.

The race-type distribution is dominated by Flat racing, but this may reflect
the affected jurisdictions rather than a race-code convention.

Seven affected source course labels did not join to the governed course
reference:

- Bordeaux Le Bouscat;
- Chukyo;
- Cidade Jardim;
- Hipodromo Chile;
- Les Landes;
- Monterrico; and
- Nakayama.

Their jurisdictions must remain unresolved in this analysis until the reference
mapping is reconciled.

The next step tests whether blank and zero values normally describe an entire
race or appear alongside positive numbers within the same race.

In [16]:
# Build one record per provisional race containing at least one blank or zero
# `num` value.
#
# We count all three observable number states:
#
# - blank text;
# - integer zero; and
# - positive integer.
#
# This reveals whether the non-positive state applies to the whole race or only
# to selected runner rows.

nonpositive_num_race_profile = pd.read_sql_query(
    """
    WITH race_num_states AS (
        SELECT
            date,
            course,
            off,
            MIN(race_id) AS race_id,
            MIN(race_name) AS race_name,
            MIN(type) AS race_type,
            MAX(ran) AS reported_ran,
            COUNT(*) AS source_runner_rows,

            SUM(
                typeof(num) = 'text'
                AND trim(CAST(num AS TEXT)) = ''
            ) AS blank_num_rows,

            SUM(
                typeof(num) = 'integer'
                AND num = 0
            ) AS zero_num_rows,

            SUM(
                typeof(num) = 'integer'
                AND num > 0
            ) AS positive_num_rows

        FROM data
        WHERE rowid <> 1
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        *
    FROM race_num_states
    WHERE
        blank_num_rows > 0
        OR zero_num_rows > 0
    ORDER BY
        date,
        course,
        off
    """,
    connection,
)

# Assign a race-level number-coverage pattern.
#
# The labels remain descriptive. They do not yet claim that blank or zero means
# missing, invalid or unnumbered.

def classify_num_race_pattern(row):
    if row["blank_num_rows"] == row["source_runner_rows"]:
        return "all_rows_blank"

    if row["zero_num_rows"] == row["source_runner_rows"]:
        return "all_rows_zero"

    if (
        row["blank_num_rows"] > 0
        and row["positive_num_rows"] > 0
        and row["zero_num_rows"] == 0
    ):
        return "mixed_blank_and_positive"

    if (
        row["zero_num_rows"] > 0
        and row["positive_num_rows"] > 0
        and row["blank_num_rows"] == 0
    ):
        return "mixed_zero_and_positive"

    if (
        row["blank_num_rows"] > 0
        and row["zero_num_rows"] > 0
        and row["positive_num_rows"] == 0
    ):
        return "mixed_blank_and_zero"

    if (
        row["blank_num_rows"] > 0
        and row["zero_num_rows"] > 0
        and row["positive_num_rows"] > 0
    ):
        return "mixed_blank_zero_and_positive"

    return "other_pattern"


nonpositive_num_race_profile["num_race_pattern"] = (
    nonpositive_num_race_profile.apply(
        classify_num_race_pattern,
        axis=1,
    )
)

# Summarise the race-level patterns.
#
# Runner-row totals are retained alongside race counts so that a small number
# of large races cannot be mistaken for a widespread convention.

nonpositive_num_race_pattern_summary = (
    nonpositive_num_race_profile
    .groupby(
        "num_race_pattern",
        dropna=False,
    )
    .agg(
        provisional_races=("date", "size"),
        source_runner_rows=("source_runner_rows", "sum"),
        blank_num_rows=("blank_num_rows", "sum"),
        zero_num_rows=("zero_num_rows", "sum"),
        positive_num_rows=("positive_num_rows", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
    .sort_values(
        "provisional_races",
        ascending=False,
    )
)

display(nonpositive_num_race_pattern_summary)

,num_race_pattern,provisional_races,source_runner_rows,blank_num_rows,zero_num_rows,positive_num_rows,first_date,last_date
0,all_rows_blank,863,6952,6952,0,0,2015-01-01,2026-05-10
1,all_rows_zero,174,1157,0,1157,0,2015-02-28,2026-03-21
3,mixed_blank_and_zero,8,84,66,18,0,2016-12-28,2025-10-09
2,mixed_blank_and_positive,6,53,14,0,39,2015-02-21,2023-08-26
4,mixed_zero_and_positive,4,23,0,4,19,2017-08-28,2022-01-02


### Race-level pattern of blank and zero values

Blank and zero runner-number values are overwhelmingly race-wide source states.

Of the 1,055 provisional races containing at least one blank or zero `num`:

- 863 races have blank `num` on every stored runner row;
- 174 races have integer zero on every stored runner row;
- 8 races contain both blank and zero values but no positive numbers;
- 6 races mix blank values with positive numbers; and
- 4 races mix zero values with positive numbers.

This means non-positive `num` values are usually properties of the race's source
representation rather than isolated runner-level omissions.

However, blank text and integer zero must still remain separate raw states:

- they have different jurisdictional distributions;
- they can coexist within the same race; and
- the source provides no evidence that they are semantically interchangeable.

Safe provisional treatment:

- preserve the raw storage state;
- derive a nullable canonical runner number only from positive integers;
- retain a separate status for blank text and integer zero;
- classify race-level number coverage independently;
- inspect mixed-state races as exceptions rather than generalising from them.

In [18]:
# Isolate the small set of races containing more than one runner-number state.
#
# These are the only cases where blank or zero values appear alongside another
# state within the same provisional race.

mixed_num_races = (
    nonpositive_num_race_profile.loc[
        nonpositive_num_race_profile["num_race_pattern"].str.startswith("mixed_")
    ]
    .copy()
    .sort_values(
        [
            "num_race_pattern",
            "date",
            "course",
            "off",
        ]
    )
    .reset_index(drop=True)
)

display(
    mixed_num_races[
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "race_type",
            "reported_ran",
            "source_runner_rows",
            "blank_num_rows",
            "zero_num_rows",
            "positive_num_rows",
            "num_race_pattern",
        ]
    ]
)

# Load only the runner rows from those mixed-state races.
#
# Filtering is performed in SQLite through a temporary key table so that the
# full source is not loaded into pandas.

connection.execute(
    """
    CREATE TEMP TABLE IF NOT EXISTS mixed_num_race_keys_temp (
        date TEXT NOT NULL,
        course TEXT NOT NULL,
        off TEXT NOT NULL,
        PRIMARY KEY (date, course, off)
    )
    """
)

connection.execute("DELETE FROM mixed_num_race_keys_temp")

connection.executemany(
    """
    INSERT INTO mixed_num_race_keys_temp (
        date,
        course,
        off
    )
    VALUES (?, ?, ?)
    """,
    list(
        mixed_num_races[
            [
                "date",
                "course",
                "off",
            ]
        ].itertuples(index=False, name=None)
    ),
)

mixed_num_runner_rows = pd.read_sql_query(
    """
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.off,
        d.race_id,
        d.race_name,
        d.type AS race_type,
        d.ran,
        d.num,
        typeof(d.num) AS num_storage_class,
        d.draw,
        d.pos,
        d.horse,
        d.jockey,
        d.trainer,
        d.owner,
        d.sp,
        d.comment
    FROM data AS d
    INNER JOIN mixed_num_race_keys_temp AS k
        ON d.date = k.date
        AND d.course = k.course
        AND d.off = k.off
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        CASE
            WHEN typeof(d.num) = 'text' THEN -1
            ELSE d.num
        END,
        CASE
            WHEN typeof(d.pos) = 'integer' THEN d.pos
            ELSE 999
        END,
        d.rowid
    """,
    connection,
)

display(mixed_num_runner_rows)

,date,course,off,race_id,race_name,race_type,reported_ran,source_runner_rows,blank_num_rows,zero_num_rows,positive_num_rows,num_race_pattern
0,2015-02-21,Gulfstream Park (USA),10:30,619838,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,8,5,0,3,mixed_blank_and_positive
1,2015-06-15,Auteuil (FR),12:30,629751,Grand Prix Arsene Saupiquet (Prix Burgrave II) (Hurdle) (Claimer) (3yo Fillies) (Turf),Hurdle,15,15,1,0,14,mixed_blank_and_positive
2,2017-09-27,Auteuil (FR),3:10,685641,Prix Grandlieu (Hurdle) (Conditions) (5yo) (Turf),Hurdle,8,8,2,0,6,mixed_blank_and_positive
3,2018-03-22,Fontainebleau (FR),4:35,697571,Prix du Perigord (Handicap) (5yo+) (Turf),Flat,10,10,2,0,8,mixed_blank_and_positive
4,2018-09-01,Baden-Baden (GER),1:40,711064,Sport-Welt Steher Cup () (3yo+) (Turf),Flat,6,6,2,0,4,mixed_blank_and_positive
5,2023-08-26,Baden-Baden (GER),1:50,848387,Baden-Badener Steher Cup () (3yo+) (Turf),Flat,6,6,2,0,4,mixed_blank_and_positive
6,2016-12-28,Sonoda (JPN),11:07,666689,Hyogo Gold Trophy (LocalHandicap) (3yo+) (Dirt),Flat,12,12,11,1,0,mixed_blank_and_zero
7,2017-10-12,Mombetsu (JPN),11:07,687374,Edelweiss Sho (Local (Fillies) (Dirt),Flat,16,16,14,2,0,mixed_blank_and_zero
8,2018-01-24,Ohi (JPN),11:07,693423,TCK Jo-o Hai (Local (Fillies & Mares) (Dirt),Flat,14,14,9,5,0,mixed_blank_and_zero
9,2018-07-23,Del Mar (USA),12:39,715242,Wickerr Stakes (Turf),Flat,12,12,6,6,0,mixed_blank_and_zero


,source_rowid,date,course,off,race_id,race_name,race_type,ran,num,num_storage_class,draw,pos,horse,jockey,trainer,owner,sp,comment
0,16970,2015-02-21,Gulfstream Park (USA),10:30,619838,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,,text,5,1,Itsaknockout (USA),Luis Saez,Todd Pletcher,Starlight Racing,54/10,Finished 2nd - awarded the race
1,16967,2015-02-21,Gulfstream Park (USA),10:30,619838,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,,text,7,2,Upstart (USA),Jose L Ortiz,Richard Violette Jr,Ralph M Evans,9/10F,Finished 1st - disqualified and placed 2nd
2,16972,2015-02-21,Gulfstream Park (USA),10:30,619838,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,,text,6,4,Frosted (USA),Irad Ortiz Jr,Kiaran McLaughlin,Godolphin Racing Llc,19/5,Soon tracking leader on outer - led over 3f out - 2 lengths clear approaching 1 1/2f out - heading approaching final...
3,16974,2015-02-21,Gulfstream Park (USA),10:30,619838,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,,text,4,5,Gorgeous Bird (USA),Joel Rosario,Ian R Wilkes,Marylou Whitney Stables Llc,66/10,
4,16976,2015-02-21,Gulfstream Park (USA),10:30,619838,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,,text,8,8,Danny Boy (USA),Corey J Lanerie,Dale Romans,Donegal Racing,225/10,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,1747021,2025-10-09,Ohi (JPN),11:07,905803,Tokyo Hai (Local (Dirt),Flat,16,,text,13,11,Exult (JPN),Ryo Nobata,Katsunori Arayama,Yoichi Aoyama,,
156,1747023,2025-10-09,Ohi (JPN),11:07,905803,Tokyo Hai (Local (Dirt),Flat,16,,text,11,13,Mars Indy (JPN),Genki Fujimoto,Kazumasa Sakamoto,Godolphin,,
157,1747024,2025-10-09,Ohi (JPN),11:07,905803,Tokyo Hai (Local (Dirt),Flat,16,,text,14,14,Theurgist (USA),Takayuki Yano,Hidemitsu Sakai,Win Co Ltd,,
158,1747025,2025-10-09,Ohi (JPN),11:07,905803,Tokyo Hai (Local (Dirt),Flat,16,,text,5,15,Win Librement (JPN),Kenji Okamura,Tsutomu Ogata,Takaya Shimakawa,,


### Mixed-state races

Only 18 provisional races mix more than one `num` state.

The mixed cases do not reveal one universal convention:

- six races mix blank text with positive integers;
- eight mix blank text with integer zero;
- four mix integer zero with positive integers.

The inspected examples include different jurisdictions, periods and race types.
They therefore cannot safely be reduced to one interpretation such as coupled
entries, withdrawals or unnumbered runners.

One mixed race is the already-known 2025 Tokyo Hai, where the source is also
missing one runner row. This demonstrates that a mixed number state can coexist
with broader source incompleteness.

Governed conclusion:

- blank text and integer zero are usually race-wide source representations;
- mixed states are rare exceptions;
- mixed-state rows must not be assigned reconstructed runner numbers;
- zero must not be treated as runner number 0;
- blank and zero should both produce a null canonical runner number while
  retaining distinct raw-state statuses;
- mixed-state races should receive an explicit ambiguous coverage status rather
  than being silently normalised.

In [19]:
# Test the established candidate runner-record identity:
#
#     provisional race identity + horse
#
# Notebook 03 found this to be the leading runner-record key. Here we examine
# whether repeated horse labels within one provisional race are associated with
# multiple source numbers or duplicate physical rows.

horse_within_race_profile = pd.read_sql_query(
    """
    SELECT
        date,
        course,
        off,
        horse,
        COUNT(*) AS source_runner_rows,
        COUNT(DISTINCT num) AS distinct_raw_num_values,
        COUNT(
            DISTINCT CASE
                WHEN typeof(num) = 'integer' AND num > 0
                THEN num
            END
        ) AS distinct_positive_num_values,
        MIN(race_id) AS minimum_race_id,
        MAX(race_id) AS maximum_race_id,
        MIN(race_name) AS race_name,
        MIN(type) AS race_type
    FROM data
    WHERE rowid <> 1
    GROUP BY
        date,
        course,
        off,
        horse
    HAVING
        COUNT(*) > 1
        OR COUNT(DISTINCT num) > 1
    ORDER BY
        source_runner_rows DESC,
        date,
        course,
        off,
        horse
    """,
    connection,
)

horse_within_race_summary = pd.DataFrame(
    {
        "measure": [
            "affected race-and-horse groups",
            "affected provisional races",
            "affected source rows",
            "groups with multiple positive num values",
            "maximum rows for one horse within race",
        ],
        "value": [
            len(horse_within_race_profile),
            horse_within_race_profile[
                ["date", "course", "off"]
            ].drop_duplicates().shape[0],
            int(horse_within_race_profile["source_runner_rows"].sum()),
            int(
                (
                    horse_within_race_profile[
                        "distinct_positive_num_values"
                    ] > 1
                ).sum()
            ),
            (
                int(horse_within_race_profile["source_runner_rows"].max())
                if not horse_within_race_profile.empty
                else 0
            ),
        ],
    }
)

display(horse_within_race_summary)
display(horse_within_race_profile.head(50))

,measure,value
0,affected race-and-horse groups,0
1,affected provisional races,0
2,affected source rows,0
3,groups with multiple positive num values,0
4,maximum rows for one horse within race,0


,date,course,off,horse,source_runner_rows,distinct_raw_num_values,distinct_positive_num_values,minimum_race_id,maximum_race_id,race_name,race_type


## Stage 7 — Runner identity and `num`

The full-source test found no provisional race in which the same `horse` value
appears more than once.

There are:

- zero repeated race-and-horse groups;
- zero races containing the same horse under multiple positive `num` values;
- zero duplicate runner rows under the leading candidate identity.

This confirms the Notebook 03 finding that:

`date + course + off + horse`

remains the leading natural candidate runner-record identity for this source.

`num` adds useful source metadata, but it is not required to distinguish runner
records and must not replace `horse` in the candidate identity.

Governed conclusion:

- retain candidate race identity + `horse` as the leading natural runner key;
- preserve `num` as a source attribute;
- do not enforce uniqueness on `num` within a race;
- do not treat duplicate `num` values as duplicate runners;
- do not redesign the final runner key inside Notebook 14.

## Stage 8 — Safe staging representation

The investigation supports a conservative staging design.

### `ran`

`ran` is structurally clean and internally consistent:

- every stored value is an integer from 1 to 40;
- every runner row within a provisional race carries the same value;
- 189,038 races have source row count equal to `ran`;
- five races have fewer source rows than `ran`;
- no race has more source rows than `ran`.

However, external checks showed that internal agreement does not prove complete
field coverage. At least two Japanese races have a source `ran` value that is
lower than the published number of runners.

The field should therefore be stored as a source-presented race count rather
than renamed as an unqualified starter count.

### `num`

`num` has three raw representations:

- positive integer;
- integer zero;
- blank text.

Positive integers are not universally unique within a race. In several
jurisdictions they can represent coupled betting interests shared by multiple
horses. Elsewhere, duplicate positive values can be ambiguous source
collisions.

Blank text and integer zero are usually race-wide source states, but mixed
races also exist. Neither state should be interpreted as runner number zero or
used to reconstruct a missing number.

### Proposed staging fields

Race-level fields:

- `source_reported_ran`
- `source_runner_row_count`
- `source_ran_consistency_status`
- `source_row_count_vs_ran_status`
- `source_field_coverage_status`

Runner-level fields:

- `source_num_raw`
- `source_num_storage_class`
- `source_positive_runner_number`
- `source_num_state`
- `source_num_within_race_multiplicity`
- `source_num_uniqueness_status`

These are staging and governance fields. They do not redesign the final race or
runner keys.

In [22]:
# Record the provisional governance decisions reached by Notebook 14.
#
# This table separates:
#
# - the raw source field;
# - the safe canonical or staging representation;
# - the status needed to preserve uncertainty; and
# - rules that must not be imposed.
#
# The table is still an analytical notebook output. It will later guide the
# reusable module, tests, validator and database-integration document.

runner_entry_governance = pd.DataFrame(
    [
        {
            "area": "reported race count",
            "source_field": "ran",
            "safe_representation": "source_reported_ran",
            "status_or_rule": "preserve integer source value",
            "provisional_constraint": "integer between 1 and 40",
            "must_not_assume": "unqualified starter count or complete field size",
        },
        {
            "area": "within-race ran consistency",
            "source_field": "ran",
            "safe_representation": "source_ran_consistency_status",
            "status_or_rule": (
                "consistent when every stored runner row in the provisional "
                "race carries one ran value"
            ),
            "provisional_constraint": (
                "current full-source baseline: all 189,043 races consistent"
            ),
            "must_not_assume": (
                "consistency proves external correctness"
            ),
        },
        {
            "area": "source row coverage",
            "source_field": "ran",
            "safe_representation": "source_row_count_vs_ran_status",
            "status_or_rule": (
                "equal, below, above, or not comparable because ran conflicts"
            ),
            "provisional_constraint": (
                "current baseline: 189,038 equal; 5 below; 0 above"
            ),
            "must_not_assume": (
                "equal means the published field is complete"
            ),
        },
        {
            "area": "external field coverage",
            "source_field": "ran",
            "safe_representation": "source_field_coverage_status",
            "status_or_rule": (
                "unverified, known partial, externally contradicted, "
                "or externally verified"
            ),
            "provisional_constraint": (
                "do not derive verified completeness from source fields alone"
            ),
            "must_not_assume": (
                "all internally complete-looking races are externally complete"
            ),
        },
        {
            "area": "raw runner number",
            "source_field": "num",
            "safe_representation": "source_num_raw",
            "status_or_rule": "preserve original SQLite value and storage class",
            "provisional_constraint": (
                "observed states: positive integer, integer zero, blank text"
            ),
            "must_not_assume": (
                "blank and zero are interchangeable"
            ),
        },
        {
            "area": "canonical runner number",
            "source_field": "num",
            "safe_representation": "source_positive_runner_number",
            "status_or_rule": (
                "populate only when num is an integer greater than zero"
            ),
            "provisional_constraint": "nullable integer between 1 and 40",
            "must_not_assume": (
                "value is unique within race or bounded by ran"
            ),
        },
        {
            "area": "runner-number state",
            "source_field": "num",
            "safe_representation": "source_num_state",
            "status_or_rule": (
                "positive_integer, integer_zero, or blank_text"
            ),
            "provisional_constraint": (
                "retain raw-state distinction through ingestion"
            ),
            "must_not_assume": (
                "zero is a valid runner numbered 0"
            ),
        },
        {
            "area": "duplicate positive numbers",
            "source_field": "num",
            "safe_representation": "source_num_uniqueness_status",
            "status_or_rule": (
                "unique_within_race, shared_positive_num, or nonpositive_state"
            ),
            "provisional_constraint": (
                "current baseline: 523 shared positive-number groups "
                "across 362 races"
            ),
            "must_not_assume": (
                "shared number means duplicate runner or confirmed coupling"
            ),
        },
        {
            "area": "runner identity",
            "source_field": "horse and num",
            "safe_representation": "candidate race identity + horse",
            "status_or_rule": (
                "retain as leading natural candidate runner-record identity"
            ),
            "provisional_constraint": (
                "current baseline: zero repeated race-and-horse groups"
            ),
            "must_not_assume": (
                "race identity + num identifies one horse"
            ),
        },
    ]
)

display(runner_entry_governance)

,area,source_field,safe_representation,status_or_rule,provisional_constraint,must_not_assume
0,reported race count,ran,source_reported_ran,preserve integer source value,integer between 1 and 40,unqualified starter count or complete field size
1,within-race ran consistency,ran,source_ran_consistency_status,consistent when every stored runner row in the provisional race carries one ran value,"current full-source baseline: all 189,043 races consistent",consistency proves external correctness
2,source row coverage,ran,source_row_count_vs_ran_status,"equal, below, above, or not comparable because ran conflicts","current baseline: 189,038 equal; 5 below; 0 above",equal means the published field is complete
3,external field coverage,ran,source_field_coverage_status,"unverified, known partial, externally contradicted, or externally verified",do not derive verified completeness from source fields alone,all internally complete-looking races are externally complete
4,raw runner number,num,source_num_raw,preserve original SQLite value and storage class,"observed states: positive integer, integer zero, blank text",blank and zero are interchangeable
5,canonical runner number,num,source_positive_runner_number,populate only when num is an integer greater than zero,nullable integer between 1 and 40,value is unique within race or bounded by ran
6,runner-number state,num,source_num_state,"positive_integer, integer_zero, or blank_text",retain raw-state distinction through ingestion,zero is a valid runner numbered 0
7,duplicate positive numbers,num,source_num_uniqueness_status,"unique_within_race, shared_positive_num, or nonpositive_state",current baseline: 523 shared positive-number groups across 362 races,shared number means duplicate runner or confirmed coupling
8,runner identity,horse and num,candidate race identity + horse,retain as leading natural candidate runner-record identity,current baseline: zero repeated race-and-horse groups,race identity + num identifies one horse


### Refinement of external-status fields

External checking revealed that runner-row completeness and the correctness of
`ran` are related but separate questions.

For example:

- a race may have missing runner rows while `ran` remains externally correct;
- a race may have missing runner rows and an externally contradicted `ran`;
- a race may be externally verified as complete; or
- no external verification may have been performed.

One combined field would obscure these distinctions.

The staging model should therefore use:

#### `source_runner_coverage_status`

- `unverified`
- `internally_equal_to_ran`
- `known_partial`
- `externally_verified_complete`

#### `source_ran_external_status`

- `unverified`
- `externally_verified`
- `externally_contradicted`

`internally_equal_to_ran` remains an internal source observation. It does not
mean externally verified completeness.

In [24]:
# Replace the combined external-coverage decision with two independent status
# fields: one for stored runner coverage and one for external validation of
# `ran`.

runner_entry_governance = runner_entry_governance.loc[
    runner_entry_governance["area"] != "external field coverage"
].copy()

external_governance_rows = pd.DataFrame(
    [
        {
            "area": "stored runner coverage",
            "source_field": "ran and source row count",
            "safe_representation": "source_runner_coverage_status",
            "status_or_rule": (
                "unverified, internally_equal_to_ran, known_partial, "
                "or externally_verified_complete"
            ),
            "provisional_constraint": (
                "internal equality is not external proof of completeness"
            ),
            "must_not_assume": (
                "source rows equal to ran means the full published field "
                "is present"
            ),
        },
        {
            "area": "external validation of ran",
            "source_field": "ran",
            "safe_representation": "source_ran_external_status",
            "status_or_rule": (
                "unverified, externally_verified, or externally_contradicted"
            ),
            "provisional_constraint": (
                "populate only from recorded external evidence"
            ),
            "must_not_assume": (
                "a structurally valid and repeated ran value is externally "
                "correct"
            ),
        },
    ]
)

runner_entry_governance = (
    pd.concat(
        [
            runner_entry_governance,
            external_governance_rows,
        ],
        ignore_index=True,
    )
)

display(runner_entry_governance)

,area,source_field,safe_representation,status_or_rule,provisional_constraint,must_not_assume
0,reported race count,ran,source_reported_ran,preserve integer source value,integer between 1 and 40,unqualified starter count or complete field size
1,within-race ran consistency,ran,source_ran_consistency_status,consistent when every stored runner row in the provisional race carries one ran value,"current full-source baseline: all 189,043 races consistent",consistency proves external correctness
2,source row coverage,ran,source_row_count_vs_ran_status,"equal, below, above, or not comparable because ran conflicts","current baseline: 189,038 equal; 5 below; 0 above",equal means the published field is complete
3,raw runner number,num,source_num_raw,preserve original SQLite value and storage class,"observed states: positive integer, integer zero, blank text",blank and zero are interchangeable
4,canonical runner number,num,source_positive_runner_number,populate only when num is an integer greater than zero,nullable integer between 1 and 40,value is unique within race or bounded by ran
5,runner-number state,num,source_num_state,"positive_integer, integer_zero, or blank_text",retain raw-state distinction through ingestion,zero is a valid runner numbered 0
6,duplicate positive numbers,num,source_num_uniqueness_status,"unique_within_race, shared_positive_num, or nonpositive_state",current baseline: 523 shared positive-number groups across 362 races,shared number means duplicate runner or confirmed coupling
7,runner identity,horse and num,candidate race identity + horse,retain as leading natural candidate runner-record identity,current baseline: zero repeated race-and-horse groups,race identity + num identifies one horse
8,stored runner coverage,ran and source row count,source_runner_coverage_status,"unverified, internally_equal_to_ran, known_partial, or externally_verified_complete",internal equality is not external proof of completeness,source rows equal to ran means the full published field is present
9,external validation of ran,ran,source_ran_external_status,"unverified, externally_verified, or externally_contradicted",populate only from recorded external evidence,a structurally valid and repeated ran value is externally correct


## Stage 9 — Conclusions and limitations

### Conclusions

#### `ran`

The source field `ran` is structurally clean:

- all 1,851,285 stored values are integers;
- observed values range from 1 to 40;
- every provisional race carries one consistent `ran` value across its stored
  runner rows.

For 189,038 of 189,043 provisional races, the stored runner-row count equals
`ran`.

Five races contain fewer stored runner rows than `ran`. These are known partial
source records.

However, external checks also showed that internal equality does not prove that
the published field is complete. Some races can have:

- internally consistent `ran`;
- stored rows equal to `ran`; and
- an externally contradicted field size.

The safe interpretation is therefore:

> `ran` is a source-presented race-level count.

It must not be renamed or treated as an unqualified starter count without
external validation.

#### `num`

The source field `num` has three observed raw states:

- positive integer;
- integer zero;
- blank text.

A canonical positive runner number can be derived only when `num` is an integer
greater than zero.

Positive `num` is not universally unique within a race:

- 523 duplicated positive-number groups occur;
- 362 provisional races are affected;
- 1,084 runner rows are involved;
- up to four horses can share one positive `num`.

Some duplicated values are consistent with coupled or bracketed betting
interests. Others appear to be ambiguous source-number collisions. Duplicate
`num` alone therefore cannot establish coupling, duplication or error.

Blank and zero values are overwhelmingly race-wide source states:

- 863 races have all rows blank;
- 174 races have all rows zero;
- only 18 races mix states.

Blank text and integer zero must remain distinct raw states. Neither should be
treated as a valid runner number or used to reconstruct a missing number.

#### Runner identity

No provisional race contains the same `horse` value more than once.

The leading candidate runner-record identity remains:

`date + course + off + horse`

`num` is a governed source attribute and not a runner key.

### Limitations

This notebook cannot establish a universal sporting definition for `ran`.

The source alone does not reveal whether `ran` was intended to represent:

- declared runners;
- actual starters;
- result rows;
- a provider-specific field count; or
- another jurisdiction-dependent race count.

External checks were limited to a small set of identified exceptions. They
demonstrate that internal consistency is insufficient, but they do not provide
full external validation of all 189,043 races.

The source does not retain coupled-entry suffixes such as `1A` or `1B`.
Therefore, lost suffixes must not be reconstructed from duplicated integer
values.

Course-jurisdiction analysis depends on the governed Notebook 12 reference.
Several affected raw course labels remain unresolved and require separate
reference maintenance.

The notebook governs preservation, canonicalisation and status assignment. It
does not attempt to repair source race fields or infer missing runners.

In [25]:
# Record the final Notebook 14 decisions in a compact closeout table.
#
# These decisions are intended to drive the reusable module, unit tests,
# independent validator and database-integration documentation.

runner_counts_numbers_decisions = pd.DataFrame(
    [
        {
            "area": "ran meaning",
            "decision": "Preserve as source_reported_ran",
            "status": "Confirmed",
            "reason": (
                "Structurally consistent but not proven to equal complete "
                "published starter count"
            ),
        },
        {
            "area": "ran consistency",
            "decision": (
                "Derive within-race consistency and row-count comparison "
                "statuses"
            ),
            "status": "Confirmed",
            "reason": (
                "All races have one ran value; five have fewer stored rows "
                "than ran"
            ),
        },
        {
            "area": "external coverage",
            "decision": (
                "Keep runner coverage and ran external validation as separate "
                "statuses"
            ),
            "status": "Required",
            "reason": (
                "Internal equality does not prove external completeness or "
                "correctness"
            ),
        },
        {
            "area": "positive num",
            "decision": (
                "Derive source_positive_runner_number only for integers > 0"
            ),
            "status": "Confirmed",
            "reason": (
                "Positive values are structurally valid but not universally "
                "unique within race"
            ),
        },
        {
            "area": "blank num",
            "decision": (
                "Preserve blank_text state and canonicalise runner number to null"
            ),
            "status": "Confirmed",
            "reason": (
                "Blank is usually race-wide but can occur in mixed-state races"
            ),
        },
        {
            "area": "zero num",
            "decision": (
                "Preserve integer_zero state and canonicalise runner number to null"
            ),
            "status": "Confirmed",
            "reason": (
                "Zero is a distinct source state and not a valid runner number"
            ),
        },
        {
            "area": "duplicate positive num",
            "decision": (
                "Allow shared positive numbers and classify within-race "
                "multiplicity"
            ),
            "status": "Confirmed",
            "reason": (
                "523 shared-number groups include both genuine-looking coupled "
                "entries and ambiguous collisions"
            ),
        },
        {
            "area": "runner identity",
            "decision": (
                "Retain candidate race identity + horse as leading natural key"
            ),
            "status": "Confirmed",
            "reason": (
                "No repeated race-and-horse groups occur in the full source"
            ),
        },
        {
            "area": "suffix reconstruction",
            "decision": "Do not reconstruct lost coupled-entry suffixes",
            "status": "Required",
            "reason": (
                "The integer source field does not preserve enough evidence"
            ),
        },
        {
            "area": "course reference gaps",
            "decision": (
                "Reconcile unresolved course labels through Notebook 12 "
                "reference maintenance"
            ),
            "status": "Deferred",
            "reason": (
                "Jurisdiction assignment is reference-data work, not a num "
                "parser rule"
            ),
        },
    ]
)

display(runner_counts_numbers_decisions)

,area,decision,status,reason
0,ran meaning,Preserve as source_reported_ran,Confirmed,Structurally consistent but not proven to equal complete published starter count
1,ran consistency,Derive within-race consistency and row-count comparison statuses,Confirmed,All races have one ran value; five have fewer stored rows than ran
2,external coverage,Keep runner coverage and ran external validation as separate statuses,Required,Internal equality does not prove external completeness or correctness
3,positive num,Derive source_positive_runner_number only for integers > 0,Confirmed,Positive values are structurally valid but not universally unique within race
4,blank num,Preserve blank_text state and canonicalise runner number to null,Confirmed,Blank is usually race-wide but can occur in mixed-state races
5,zero num,Preserve integer_zero state and canonicalise runner number to null,Confirmed,Zero is a distinct source state and not a valid runner number
6,duplicate positive num,Allow shared positive numbers and classify within-race multiplicity,Confirmed,523 shared-number groups include both genuine-looking coupled entries and ambiguous collisions
7,runner identity,Retain candidate race identity + horse as leading natural key,Confirmed,No repeated race-and-horse groups occur in the full source
8,suffix reconstruction,Do not reconstruct lost coupled-entry suffixes,Required,The integer source field does not preserve enough evidence
9,course reference gaps,Reconcile unresolved course labels through Notebook 12 reference maintenance,Deferred,"Jurisdiction assignment is reference-data work, not a num parser rule"
